In [ ]:
import numpy as np
import scipy.io as sio
import time
import matplotlib.pyplot as plt

# 수정한 MP-PCA 코드
# 파일명이 newlocalpca.py라면 그대로 사용
from newlocalpca import mppca


# ============================================================
# 1. Parameter
# ============================================================

PATCH_RADIUS = 3          # 7 x 7 x 7 patch
NOISE_LEVEL = 10          # 사용할 noisy data
SLICE_IDX = None


# ============================================================
# 2. Load Original Data
# ============================================================

orig_mat = sio.loadmat("meas_gre_dir1.mat")

meas_gre = orig_mat["meas_gre"]               # (X, Y, Z, 6), complex
mask_brain = orig_mat["mask_brain"].astype(bool)

X, Y, Z, Ne = meas_gre.shape

if SLICE_IDX is None:
    SLICE_IDX = Z // 2

print("Original data shape :", meas_gre.shape)
print("Number of echoes    :", Ne)
print("Data type            :", meas_gre.dtype)


# ============================================================
# 3. Load Noisy Real / Imaginary
# ============================================================

noise_mat = sio.loadmat(
    f"noisy_meas_gre_dir1_{NOISE_LEVEL}.mat"
)

noisy_real = noise_mat["noisy_real"].astype(np.float32)
noisy_imag = noise_mat["noisy_imag"].astype(np.float32)

print("\nNoisy Real shape :", noisy_real.shape)
print("Noisy Imag shape :", noisy_imag.shape)


# ============================================================
# 4. Real + Imaginary Combined Matrix
# ============================================================

# Real 6 channels + Imaginary 6 channels
#
# (X, Y, Z, 6) + (X, Y, Z, 6)
#                  ↓
#          (X, Y, Z, 12)

multi_ch = np.concatenate(
    [noisy_real, noisy_imag],
    axis=-1
)

print("\nCombined data shape :", multi_ch.shape)
print("Expected shape      :", (X, Y, Z, 2 * Ne))


# ============================================================
# 5. Complex-style MP-PCA
#    실제로는 12-channel REAL matrix에 MP-PCA 적용
# ============================================================

print("\n▶ Running Real + Imag Combined MP-PCA...")

start_time = time.time()

denoised_all = mppca(
    multi_ch,
    mask=mask_brain,
    patch_radius=PATCH_RADIUS,
    pca_method="svd",
    return_sigma=False
)

elapsed = time.time() - start_time

print(f"✔ MP-PCA completed : {elapsed:.1f} sec")
print("Denoised shape     :", denoised_all.shape)


# ============================================================
# 6. Separate Real / Imaginary
# ============================================================

# 앞 6개 channel = Real
real_denoised = denoised_all[..., :Ne]

# 뒤 6개 channel = Imaginary
imag_denoised = denoised_all[..., Ne:]


print("\nReal denoised shape :", real_denoised.shape)
print("Imag denoised shape :", imag_denoised.shape)


# ============================================================
# 7. Reconstruct Complex MRI
# ============================================================

complex_denoised = (
    real_denoised
    + 1j * imag_denoised
)

print("Complex denoised shape :", complex_denoised.shape)
print("Complex data type      :", complex_denoised.dtype)


# ============================================================
# 8. Magnitude / Phase
# ============================================================

mag_original = np.abs(meas_gre)

mag_noisy = np.sqrt(
    noisy_real**2
    + noisy_imag**2
)

mag_denoised = np.abs(complex_denoised)

phase_original = np.angle(meas_gre)
phase_denoised = np.angle(complex_denoised)


# ============================================================
# 9. Simple Evaluation
# ============================================================

EPS = 1e-12


def snr_rms(reference, test, mask):
    """
    Reference와 test 차이를 noise로 보는 RMS 기반 SNR
    """
    ref_roi = reference[mask]
    test_roi = test[mask]

    signal = np.sqrt(
        np.mean(ref_roi**2)
    )

    noise = np.sqrt(
        np.mean((test_roi - ref_roi)**2)
    )

    return 20 * np.log10(
        signal / (noise + EPS)
    )


print("\n==============================")
print(" Echo-wise SNR")
print("==============================")

for echo in range(Ne):

    snr_noisy = snr_rms(
        mag_original[..., echo],
        mag_noisy[..., echo],
        mask_brain
    )

    snr_denoised = snr_rms(
        mag_original[..., echo],
        mag_denoised[..., echo],
        mask_brain
    )

    print(
        f"Echo {echo} | "
        f"Noisy = {snr_noisy:.2f} dB | "
        f"Denoised = {snr_denoised:.2f} dB | "
        f"Δ = {snr_denoised - snr_noisy:.2f} dB"
    )


# ============================================================
# 10. Phase Difference
# ============================================================

# 단순 subtraction은 ±pi wrap 문제가 있으므로
# circular phase difference 사용

phase_diff = np.angle(
    np.exp(
        1j * (
            phase_denoised
            - phase_original
        )
    )
)

mean_phase_error = np.mean(
    np.abs(phase_diff[mask_brain])
)

print(
    f"\nMean absolute phase error "
    f"= {mean_phase_error:.6f} rad"
)


# ============================================================
# 11. Visualization
# ============================================================

vmin, vmax = np.percentile(
    mag_original[mask_brain],
    (1, 99)
)

for echo in range(Ne):

    fig, axes = plt.subplots(
        1, 4,
        figsize=(18, 5)
    )

    fig.suptitle(
        f"Real + Imag Combined MP-PCA | "
        f"Noise {NOISE_LEVEL}% | "
        f"Echo {echo}",
        fontsize=16
    )

    # Original
    axes[0].imshow(
        mag_original[:, :, SLICE_IDX, echo],
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    axes[0].set_title("Original")
    axes[0].axis("off")

    # Noisy
    axes[1].imshow(
        mag_noisy[:, :, SLICE_IDX, echo],
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    axes[1].set_title("Noisy")
    axes[1].axis("off")

    # Denoised
    axes[2].imshow(
        mag_denoised[:, :, SLICE_IDX, echo],
        cmap="gray",
        vmin=vmin,
        vmax=vmax
    )
    axes[2].set_title("Denoised Complex")
    axes[2].axis("off")

    # Difference
    diff = (
        mag_denoised[:, :, SLICE_IDX, echo]
        - mag_original[:, :, SLICE_IDX, echo]
    )

    diff_max = np.percentile(
        np.abs(diff),
        99
    )

    axes[3].imshow(
        diff,
        cmap="bwr",
        vmin=-diff_max,
        vmax=diff_max
    )
    axes[3].set_title("Denoised - Original")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()


# ============================================================
# 12. Save Result
# ============================================================

output_name = (
    f"denoised_complex_combined_"
    f"noise{NOISE_LEVEL}_patch{PATCH_RADIUS}.mat"
)

sio.savemat(
    output_name,
    {
        "complex_denoised": complex_denoised,
        "real_denoised": real_denoised,
        "imag_denoised": imag_denoised,
        "magnitude_denoised": mag_denoised,
        "phase_denoised": phase_denoised,
        "mask_brain": mask_brain
    }
)

print(f"\n✔ Saved : {output_name}")